# 03 · 掩码对比教程：IRM vs cRM
目标：理解 **幅度掩码（IRM）** 与 **复数掩码（cRM）** 的本质区别——相位能不能被修正。
配套：`core/losses.py`（`irm_target`）、`exp/03_crn/model.py`、INTERVIEW-WP02/03。

In [ ]:
import sys, torch, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '..')
from core.stft import STFT
from core.losses import irm_target
torch.manual_seed(1); sr=16000; n=sr
t=np.arange(n)/sr
clean = torch.from_numpy(np.sin(2*np.pi*300*t)+0.5*np.sin(2*np.pi*900*t)).float()[None]
noise = torch.randn(1,n)*0.3
mix = clean + noise
stft = STFT(512,256)
C, N, M = stft.transform(clean), stft.transform(noise), stft.transform(mix)
print('spectra', C.shape)

## 1. IRM：只改幅度，保留带噪相位（✏️ 改 SNR）
IRM = sqrt(P_clean/(P_clean+P_noise))，作用在幅度域；相位沿用带噪相位 → 高噪声下相位错，
听感有'电音/金属感'。这就是 cRM 存在的动机。

In [ ]:
irm = irm_target(M, C)          # (0,1) 实数掩码
enh_irm = irm * M               # 只缩放幅度，相位=M 的相位（带噪）
# cRM 的理想上限：直接恢复干净复谱（实部/虚部都能修）
crm_ideal = C / (M + 1e-8)      # 复数掩码（含相位修正）
enh_crm = crm_ideal * M         # ≈ C
err_irm = (enh_irm - C).abs().mean().item()
err_crm = (enh_crm - C).abs().mean().item()
print(f'IRM 与干净谱误差={err_irm:.4f}   cRM(理想) 误差={err_crm:.2e}')
print('=> cRM 能修相位，IRM 不能')

## 2. 可视化掩码（✏️ 改窗长看时频块）
看 IRM 掩码长什么样：语音段接近 1、噪声段接近 0 的'时频开关'。

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,3.5),sharey=True)
for ax,S,ttl in zip(axes,[M[0].abs(),C[0].abs(),irm[0]],['带噪幅度','干净幅度','IRM 掩码']):
    v = S.numpy() if ttl=='IRM 掩码' else 20*np.log10(S.numpy()+1e-6)
    ax.imshow(v,origin='lower',aspect='auto',cmap='viridis' if ttl=='IRM 掩码' else 'magma')
    ax.set_title(ttl)
plt.tight_layout(); plt.show()

## 3. 面试自检
1. 写出 IRM 公式，取值范围？为什么开方（作用在幅度域）？
2. IRM 和 cRM 的根本区别是什么？（相位）
3. 为什么 cRM 输出常接 tanh 限幅（本仓 1.55·tanh）？
4. 本仓实测 IRM 和 cRM 哪个 PESQ 高？（2.84 vs 2.53，IRM 保留相位反而更高——为什么？
   提示：VoiceBank 噪声不算极端，相位误差不是瓶颈；cRM 自由度大更难训）